# 🎰 Trabalho AC2 — Métodos de Monte Carlo em Aprendizado por Reforço

**Disciplina:** Aprendizado por Reforço — Ibmec · **Entrega:** 16/10/2026
**Grupo:** _(preencher: até 3 alunos)_
**Referência teórica:** Sutton & Barto (2018), cap. 5 · **Ambiente:** `Blackjack-v1` (Gymnasium)

Este notebook **reutiliza as funções do notebook didático do professor**
(`referencia/Aula_13_14_Monte_Carlo_RL_Didatico.ipynb`, Aulas 13–14) e as organiza para os
experimentos que o trabalho pede. Toda linha que **difere** do original está marcada com `# [ALTERADO]`
e o motivo está explicado na seção 0.

## Roteiro (espelha os 3 itens da apresentação)

| Item da apresentação | Onde está neste notebook |
|---|---|
| 1. Análise da implementação dos algoritmos | Seções 2, 4 e 5 (código comentado) + seção 8 (análise teórica) |
| 2. Desempenho das políticas implementadas | Seção 3 (First-Visit com políticas do grupo) |
| 3. Comparação de experimentos com parâmetros diferentes | Seção 6 (ES × On-Policy ε-greedy) |

> ⏱️ **Tempo de execução:** completo ≈ 15–25 min. Para um teste rápido (~10× menor), rode com a variável
> de ambiente `MC_RAPIDO=1` ou troque `MODO_RAPIDO` para `True` na célula de configuração.

## 0. O que foi alterado em relação ao notebook do professor (e por quê)

Ao ler o código original, encontramos pontos que afetam a **comparação entre os algoritmos**:

1. **Estados com soma < 12 (principal).** O ambiente gera somas de 4 a 21 (são **280 estados**: 18×10 sem ás + 10×10 com ás;
   o próprio output do notebook original mostra 280 estados visitados). Porém o ES original só sorteia somas 12–21
   e a avaliação usa `politica.get(state, 0)`, ou seja, **STICK** para qualquer estado não aprendido.
   Resultado: ES (e a política simples convertida para `dict` só com 12–21) “param” com 8, 9, 10… o que é obviamente ruim
   e **enviesa a tabela comparativa a favor do On-Policy**, que aprende esses estados.
   Correção: o ES agora sorteia todos os 280 estados e a avaliação usa `HIT` como padrão para soma < 12
   (nunca estoura). A seção 3.3 mostra a diferença numérica.
2. **Reprodutibilidade.** Sementes fixas (`np.random.default_rng`, `env.reset(seed=...)`) e vários *seeds* por experimento
   (média ± erro padrão), pois uma única rodada de MC tem muita variância.
3. **Primeira visita.** No original, o teste `if sa not in visitados` roda **de trás para frente**, o que marca a
   *última* ocorrência, não a primeira. Aqui o índice da primeira visita é calculado antes (`primeiras_visitas`).
   Curiosidade (verificada na seção 3.2): **no Blackjack nenhum estado se repete dentro de um episódio**, então
   First-Visit = Every-Visit — o Exercício 1/5 do professor dará resultados equivalentes.
4. **Métricas.** O original usa a média *não ponderada* de V(s). Aqui o desempenho é o **retorno médio por jogo** (± erro padrão),
   que é o valor real da política, além de % de vitórias/empates/derrotas.
5. **ε-greedy.** Aceita `eps` fixo ou uma função de decaimento (Exercício 4) e usa gerador aleatório com semente.
   Também corrigimos a terminologia: com prob. 1−ε é **aproveitamento** (*exploitation*), com prob. ε é **exploração**.
6. **Desempenho do código.** Soma + contador em vez de listas de retornos; check de primeira visita O(T) em vez de O(T²).

## 1. Configuração

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
from pathlib import Path
from collections import defaultdict
from typing import Callable, Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

SEED = 42
MODO_RAPIDO = os.environ.get("MC_RAPIDO", "0") == "1"   # True → experimentos ~10x menores (teste de sanidade)
SALVAR = True                                            # salva figuras/tabelas em resultados/

RES_DIR = Path("resultados") if Path("resultados").exists() else Path("../resultados")
RES_DIR.mkdir(parents=True, exist_ok=True)

VERIF = {}   # verificações de sanidade, salvas em resultados/verificacoes.csv
STICK, HIT = 0, 1
State = Tuple[int, int, bool]
Policy = Callable[[State], int]

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})


def salvar_fig(fig, nome: str):
    """Salva a figura em resultados/<nome>.png (para usar nos slides)."""
    if SALVAR:
        fig.savefig(RES_DIR / f"{nome}.png", dpi=160, bbox_inches="tight")


def salvar_csv(df: pd.DataFrame, nome: str):
    if SALVAR:
        df.to_csv(RES_DIR / f"{nome}.csv", index=False)


print(f"Gymnasium {gym.__version__} | NumPy {np.__version__} | MODO_RAPIDO={MODO_RAPIDO}")
print(f"Resultados serão salvos em: {RES_DIR.resolve()}")

## 2. Ambiente e funções-base (reaproveitadas do notebook do professor)

**Blackjack-v1:** ações `0=STICK`, `1=HIT`; estado `(soma_jogador, carta_dealer, ás_usável)`;
recompensa +1 / 0 / −1 **só no fim** do episódio; o dealer pede carta enquanto sua soma for < 17.
Como a recompensa é terminal e γ=1, **todos os estados de um episódio recebem o mesmo retorno** (o resultado do jogo).

In [ ]:
def make_env(seed=None):
    """Cria o Blackjack-v1 com semente fixa."""
    env = gym.make("Blackjack-v1")
    env.reset(seed=seed)              # [ALTERADO] semente para reprodutibilidade
    env.action_space.seed(seed)
    return env


def generate_episode(env, policy: Policy):
    """Gera um episódio completo seguindo `policy`.
    Retorna (trajetória, T) com trajetória = [(S_t, R_t, done, A_t), ...] e T = índice do estado terminal.
    (Mesmo formato do notebook original: R_t é a recompensa recebida ao CHEGAR em S_t.)"""
    trajectory = []
    done = True
    At = None
    while True:
        if done:
            St, _ = env.reset()
            Rt = None
            done = False
        else:
            St, Rt, terminated, truncated, _ = env.step(At)
            done = bool(terminated or truncated)
        At = policy(St)
        trajectory.append((St, Rt, done, At))
        if done:
            break
    return trajectory, len(trajectory) - 1


def primeiras_visitas(traj, T, por_par=False):
    """[ALTERADO] Índice da PRIMEIRA ocorrência de cada estado (ou par estado-ação) no episódio."""
    vistos = {}
    for t in range(T):
        chave = (traj[t][0], traj[t][3]) if por_par else traj[t][0]
        vistos.setdefault(chave, t)
    return vistos


def simular_politica(env, politica: Policy, n_jogos: int = 50_000) -> dict:
    """Joga `n_jogos` partidas com a política e devolve % de vitórias/empates/derrotas e o retorno médio (± erro padrão)."""
    rec = np.empty(n_jogos)
    for i in range(n_jogos):
        state, _ = env.reset()
        done = False
        while not done:
            state, reward, terminated, truncated, _ = env.step(politica(state))
            done = terminated or truncated
        rec[i] = reward
    return {
        "retorno_medio": rec.mean(),
        "erro_padrao": rec.std(ddof=1) / np.sqrt(n_jogos),
        "vitorias_%": (rec > 0).mean() * 100,
        "empates_%": (rec == 0).mean() * 100,
        "derrotas_%": (rec < 0).mean() * 100,
    }


def politica_de_dict(pi: Dict) -> Policy:
    """[ALTERADO] Converte uma política aprendida (dict) em função. Estados não aprendidos:
    HIT se soma < 12 (nunca estoura), STICK caso contrário. O original usava sempre STICK."""
    def f(state):
        if state in pi:
            return pi[state]
        return HIT if state[0] < 12 else STICK
    return f


def argmax_aleatorio(valores: List[float], rng) -> int:
    """Argmax com quebra de empates aleatória (importante quando todos os Q começam em 0)."""
    arr = np.asarray(valores, dtype=float)
    melhores = np.flatnonzero(arr == arr.max())
    return int(melhores[0]) if len(melhores) == 1 else int(rng.choice(melhores))


def q_para_v(Q: Dict) -> Dict:
    """V(s) ≈ max_a Q(s,a)."""
    V = {}
    for (s, a), q in Q.items():
        V[s] = max(V.get(s, -np.inf), q)
    return V


def derivar_politica_greedy(Q: Dict, n_acoes: int = 2, rng=None) -> Dict:
    """Política greedy (determinística) a partir de Q. Ações nunca vistas valem −∞."""
    rng = rng or np.random.default_rng(0)
    estados_acoes = defaultdict(dict)
    for (s, a), q in Q.items():
        estados_acoes[s][a] = q
    return {s: argmax_aleatorio([acoes.get(a, -np.inf) for a in range(n_acoes)], rng)
            for s, acoes in estados_acoes.items()}

### 2.1 Visualizações (heatmaps de V(s) e da política, curvas de aprendizado)

In [ ]:
def valor_para_matriz(V: Dict, usable_ace: bool) -> np.ndarray:
    """Matriz 10x10 (soma 12..21 × dealer A..10) para o heatmap. Estados sem estimativa = NaN."""
    m = np.full((10, 10), np.nan)
    for ps in range(12, 22):
        for dc in range(1, 11):
            if (ps, dc, usable_ace) in V:
                m[ps - 12, dc - 1] = V[(ps, dc, usable_ace)]
    return m


def plot_valor_heatmap(V, titulo="", mostrar_valores=False, arquivo=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig.suptitle(titulo, fontsize=13, fontweight="bold")
    for ax, ace in zip(axes, [True, False]):
        dados = valor_para_matriz(V, ace)
        im = ax.imshow(dados, cmap="viridis", origin="lower", extent=[0.5, 10.5, 11.5, 21.5],
                       aspect="auto", vmin=-1, vmax=1)
        ax.set_title("Com ás usável" if ace else "Sem ás usável")
        ax.set_xlabel("Carta visível do dealer")
        ax.set_ylabel("Soma do jogador")
        ax.set_xticks(range(1, 11))
        ax.set_xticklabels(["A"] + [str(x) for x in range(2, 11)])
        ax.set_yticks(range(12, 22))
        ax.grid(False)
        if mostrar_valores:
            for i in range(10):
                for j in range(10):
                    v = dados[i, j]
                    if not np.isnan(v):
                        ax.text(j + 1, i + 12, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                                color="white" if v < 0 else "black", fontweight="bold")
        fig.colorbar(im, ax=ax, label="V(s)")
    fig.tight_layout()
    if arquivo:
        salvar_fig(fig, arquivo)
    plt.show()


def plot_politica_heatmap(pi, titulo="Política aprendida", arquivo=None):
    """Verde = HIT, vermelho = STICK. Estados não aprendidos seguem a regra de `politica_de_dict`."""
    f = politica_de_dict(pi)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig.suptitle(titulo, fontsize=13, fontweight="bold")
    for ax, ace in zip(axes, [True, False]):
        arr = np.array([[f((ps, dc, ace)) for dc in range(1, 11)] for ps in range(12, 22)])
        im = ax.imshow(arr, origin="lower", extent=[0.5, 10.5, 11.5, 21.5], aspect="auto",
                       cmap="RdYlGn", vmin=0, vmax=1)
        ax.set_title("Com ás usável" if ace else "Sem ás usável")
        ax.set_xlabel("Carta visível do dealer")
        ax.set_ylabel("Soma do jogador")
        ax.set_xticks(range(1, 11))
        ax.set_xticklabels(["A"] + [str(x) for x in range(2, 11)])
        ax.set_yticks(range(12, 22))
        ax.grid(False)
        for i in range(10):
            for j in range(10):
                ax.text(j + 1, i + 12, "H" if arr[i, j] == HIT else "S", ha="center", va="center",
                        fontsize=9, fontweight="bold", color="white")
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_ticks([0, 1])
        cbar.set_ticklabels(["STICK", "HIT"])
    fig.tight_layout()
    if arquivo:
        salvar_fig(fig, arquivo)
    plt.show()


def plot_curvas_aprendizado(df, linhas_ref=None, titulo="", arquivo=None):
    """df: colunas label, seed, episodios, retorno. Média ± erro padrão entre seeds."""
    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    for label, g in df.groupby("label", sort=False):
        s = g.groupby("episodios")["retorno"].agg(["mean", "sem"]).fillna(0)
        ax.plot(s.index, s["mean"], marker="o", ms=4, label=label)
        ax.fill_between(s.index, s["mean"] - s["sem"], s["mean"] + s["sem"], alpha=0.12)
    if linhas_ref:
        for nome, v in linhas_ref.items():
            ax.axhline(v, ls="--", lw=1, color="gray")
            # x em coordenadas do eixo (0–1), y em dados: o rótulo fica sempre dentro do gráfico
            ax.text(0.01, v, f"{nome} ({v:+.3f})", va="bottom", fontsize=8, color="dimgray",
                    transform=ax.get_yaxis_transform())
    ax.set_xscale("log")
    ax.set_xlabel("Episódios de treino (escala log)")
    ax.set_ylabel("Retorno médio da política greedy")
    ax.set_title(titulo)
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    fig.tight_layout()
    if arquivo:
        salvar_fig(fig, arquivo)
    plt.show()

## 3. First-Visit MC Prediction — avaliando políticas

**Teoria (S&B 5.1):** para uma política fixa π, $V_\pi(s)=\mathbb{E}_\pi[G_t\mid S_t=s]$ é estimado pela **média dos retornos**
observados após a **primeira** visita a $s$ em cada episódio. O retorno é calculado de trás para frente com
$G_t = R_{t+1} + \gamma G_{t+1}$.

In [ ]:
def first_visit_MC_prediction(env, policy: Policy, episodes: int = 10_000, gamma: float = 1.0,
                              track_states=(), track_every: int = 1_000):
    """First-Visit MC Prediction.
    Retorna (V, Counts, info). `info` traz a recompensa final e o retorno G0 de cada episódio
    e, opcionalmente, a evolução de V em `track_states` (para análise de convergência)."""
    Returns_sum = defaultdict(float)     # soma dos retornos por estado
    Counts = defaultdict(int)            # nº de primeiras visitas por estado
    hist = {s: [] for s in track_states}
    R_final = np.empty(episodes)
    G0 = np.empty(episodes)

    for ep in range(episodes):
        traj, T = generate_episode(env, policy)
        primeira = primeiras_visitas(traj, T)          # [ALTERADO] 1ª visita calculada uma vez, O(T)
        G = 0.0
        for t in range(T - 1, -1, -1):                 # de trás para frente
            St = traj[t][0]
            G = gamma * G + traj[t + 1][1]             # G_t = γ·G_{t+1} + R_{t+1}
            if primeira[St] == t:                      # é a primeira visita?
                Returns_sum[St] += G
                Counts[St] += 1
        R_final[ep] = traj[T][1]
        G0[ep] = G                                     # retorno a partir do estado inicial
        if track_states and (ep + 1) % track_every == 0:
            for s in track_states:
                c = Counts.get(s, 0)
                hist[s].append((ep + 1, Returns_sum.get(s, 0.0) / c if c else np.nan))

    V = {s: Returns_sum[s] / Counts[s] for s in Counts}
    return V, dict(Counts), {"R_final": R_final, "G0": G0, "hist": hist}

### 3.1 Políticas

As duas primeiras (**`simples_20`** e a aleatória) são **referências**. As políticas do **grupo** são:

- **Imita o dealer (≥17):** copia a própria regra do dealer — compra carta enquanto a soma for menor
  que 17, para com 17 ou mais. Fácil de justificar: é literalmente a regra que o jogo já usa para o
  dealer, então serve como "piso" natural de comparação (será que jogar como o dealer é uma boa ideia
  pro jogador também?).
- **Depende da carta do dealer:** se a carta visível do dealer estiver entre **2 e 5** (mesa "fraca" —
  o dealer tem boa chance de estourar sozinho comprando carta), o jogador para cedo, com soma **≥ 12**,
  deixando o risco todo para o dealer. Com qualquer outra carta do dealer (6, 7, 8, 9, 10 ou Ás), o
  jogador segue a regra padrão e só para com soma **≥ 17**. A ideia veio de observar os heatmaps de
  política da seção 6 (Folio dos gráficos de decisão): visualmente, boa parte das políticas aprendidas
  já "para mais cedo" contra cartas baixas do dealer.

⚠️ Detalhe do ambiente: o ás do dealer aparece codificado como **1** — por isso um teste como
`dealer >= 7` trataria o ás, erradamente, como carta fraca; aqui a faixa "fraca" é testada
explicitamente como `2 <= dealer <= 5`.

In [ ]:
RNG_POL = np.random.default_rng(SEED)

def politica_simples_20(state):
    """Referência do professor (Sutton & Barto): STICK só com 20 ou 21."""
    soma, dealer, ace = state
    return STICK if soma >= 20 else HIT


def politica_aleatoria(state):
    """Baseline: 50/50."""
    return int(RNG_POL.integers(2))


# ---- ✏️ POLÍTICAS DO GRUPO --------------------------------------------------------------
def politica_imita_dealer(state):
    """Copia a regra do próprio dealer: compra carta enquanto a soma for menor que 17."""
    soma, dealer, ace = state
    return STICK if soma >= 17 else HIT


def politica_dealer_fraco(state):
    """Se a carta do dealer estiver entre 2 e 5 (mesa fraca, dealer arrisca estourar sozinho),
    o jogador para cedo (soma >= 12). Com qualquer outra carta do dealer, segue a regra padrão
    e só para com soma >= 17."""
    soma, dealer, ace = state
    if 2 <= dealer <= 5:
        return STICK if soma >= 12 else HIT
    return STICK if soma >= 17 else HIT


POLITICAS = {
    "Simples ≥20 (professor)": politica_simples_20,
    "Imita o dealer (≥17)": politica_imita_dealer,
    "Depende da carta do dealer": politica_dealer_fraco,
    "Aleatória 50/50": politica_aleatoria,
}

### 3.2 Por que First-Visit = Every-Visit no Blackjack?

Se nenhum estado se repete dentro de um episódio, filtrar “só a primeira visita” não muda nada.
Verificamos empiricamente:

In [ ]:
def verificar_repeticoes(env, policy, n=50_000):
    repetidos = 0
    for _ in range(n):
        traj, T = generate_episode(env, policy)
        estados = [traj[t][0] for t in range(T)]
        repetidos += len(estados) != len(set(estados))
    return repetidos

env_chk = make_env(SEED)
for nome in ["Aleatória 50/50", "Imita o dealer (≥17)"]:
    r = verificar_repeticoes(env_chk, POLITICAS[nome])
    VERIF[f"episodios_com_estado_repetido (de 50000) · {nome}"] = r
    print(f"{nome:<22}: episódios com estado repetido = {r} de 50.000")

### 3.3 Demonstração do viés dos estados com soma < 12

Mesma política (`simples_20`), avaliada de duas formas: (a) como `dict` apenas com os estados 12–21 (como no notebook original,
STICK nos demais) e (b) como função (HIT abaixo de 12).

In [ ]:
N_DEMO = 5_000 if MODO_RAPIDO else 50_000
estados_12_21 = [(ps, dc, ace) for ps in range(12, 22) for dc in range(1, 11) for ace in (True, False)]

env_demo = make_env(SEED + 1)
linhas_demo = []
for nome_pol, pol in [("Simples ≥20", politica_simples_20), ("Depende da carta do dealer", politica_dealer_fraco)]:
    pi_dict_original = {s: pol(s) for s in estados_12_21}                                  # só 12–21, como no original
    a = simular_politica(env_demo, lambda s: pi_dict_original.get(s, STICK), N_DEMO)      # padrão STICK (original)
    b = simular_politica(env_demo, pol, N_DEMO)                                            # função completa (HIT < 12)
    linhas_demo.append({"Política": nome_pol, "Avaliação": "dict 12–21, padrão STICK (original)", **a})
    linhas_demo.append({"Política": nome_pol, "Avaliação": "função completa (HIT < 12)", **b})
demo = pd.DataFrame(linhas_demo)[["Política", "Avaliação", "retorno_medio", "vitorias_%", "empates_%", "derrotas_%"]]
print(demo.round(3).to_string(index=False))
salvar_csv(demo, "demo_vies_somas_baixas")
# Obs.: para a `Simples ≥20` o efeito é pequeno porque ela já é muito ruim (pede carta até 19).
# O efeito grande aparece em políticas boas e no ES — veja o Experimento E (seção 6).

### 3.4 Desempenho das políticas com First-Visit MC

Para cada política: $N$ episódios, γ=1. O **retorno médio por episódio** (= $V_\pi$ da distribuição inicial) e as % de V/E/D saem dos próprios
episódios do First-Visit. Também acompanhamos a convergência de $V(s)$ em estados-chave.

In [ ]:
N_FV = 20_000 if MODO_RAPIDO else 200_000
ESTADOS_CHAVE = [(16, 10, False), (20, 6, False), (12, 1, False)]

env_fv = make_env(SEED)
V_fv, info_fv, linhas = {}, {}, []
for nome, pol in POLITICAS.items():
    t0 = time.time()
    V, cnt, info = first_visit_MC_prediction(env_fv, pol, episodes=N_FV, gamma=1.0,
                                             track_states=ESTADOS_CHAVE, track_every=max(1, N_FV // 200))
    V_fv[nome], info_fv[nome] = V, info
    r = info["R_final"]
    linhas.append({"Política": nome, "Estados visitados": len(V),
                   "Retorno médio": r.mean(), "Erro padrão": r.std(ddof=1) / np.sqrt(len(r)),
                   "Vitórias %": (r > 0).mean() * 100, "Empates %": (r == 0).mean() * 100,
                   "Derrotas %": (r < 0).mean() * 100})
    print(f"{nome:<26} {N_FV:,} episódios em {time.time() - t0:5.1f}s")

tab_fv = pd.DataFrame(linhas).sort_values("Retorno médio", ascending=False)
print()
print(tab_fv.round(4).to_string(index=False))
salvar_csv(tab_fv, "fv_desempenho_politicas")
REF_RETORNOS = {r["Política"]: r["Retorno médio"] for r in linhas}

In [ ]:
# Heatmaps de V(s) de cada política (para os slides)
for nome in POLITICAS:
    slug = "".join(c if c.isalnum() else "_" for c in nome).strip("_").lower()
    plot_valor_heatmap(V_fv[nome], titulo=f"First-Visit MC — {nome} ({N_FV:,} episódios)",
                       mostrar_valores=False, arquivo=f"fv_heatmap_{slug}")

In [ ]:
# Convergência de V(s) em estados-chave
fig, axes = plt.subplots(1, len(ESTADOS_CHAVE), figsize=(15, 4), sharex=True)
for ax, s in zip(axes, ESTADOS_CHAVE):
    for nome in POLITICAS:
        h = np.array(info_fv[nome]["hist"][s])
        if len(h):
            ax.plot(h[:, 0], h[:, 1], label=nome, lw=1.4)
    ax.set_title(f"V{s}")
    ax.set_xlabel("Episódios")
axes[0].set_ylabel("V(s) estimado")
axes[0].legend(fontsize=7)
fig.suptitle("Convergência do First-Visit MC em estados-chave", fontweight="bold")
fig.tight_layout()
salvar_fig(fig, "fv_convergencia_estados_chave")
plt.show()

## 4. Monte Carlo ES (Exploring Starts)

**Teoria (S&B 5.3):** para estimar $Q_\pi(s,a)$ com política determinística é preciso garantir que todo par $(s,a)$ seja visitado.
O ES resolve isso **começando cada episódio em um par $(S_0,A_0)$ sorteado**. Depois do primeiro passo, segue a política atual,
e após cada atualização de $Q$ a política vira gulosa: $\pi(s)\leftarrow\arg\max_a Q(s,a)$ (Generalized Policy Iteration).

O Gymnasium não permite escolher o estado inicial; o wrapper `BlackjackES` (do professor) contorna isso sobrescrevendo as mãos em `env.unwrapped`.

In [ ]:
class BlackjackES:
    """Wrapper do Blackjack que permite iniciar o episódio em qualquer estado (necessário para ES)."""

    def __init__(self, seed=None, soma_min: int = 4):
        self._env = gym.make("Blackjack-v1")
        self._env.reset(seed=seed)
        self.rng = np.random.default_rng(seed)       # [ALTERADO] gerador com semente
        self.nb_actions = 2
        # [ALTERADO] espaço completo de estados: somas 4..21 sem ás, 12..21 com ás (280 estados)
        self.estados = [(ps, dc, ace) for ace in (False, True) for ps in range(12 if ace else soma_min, 22)
                        for dc in range(1, 11)]

    def amostrar_estado_acao(self):
        """Sorteia (S0, A0) uniformemente — o 'exploring start'."""
        s = self.estados[int(self.rng.integers(len(self.estados)))]
        return s, int(self.rng.integers(2))

    def reset_es(self, estado_inicial):
        """Força o ambiente a começar em `estado_inicial` (soma, carta do dealer, ás usável)."""
        player_sum, dealer_card, has_ace = estado_inicial
        base = self._env.unwrapped
        self._env.reset()
        dealer_hides = int(self.rng.choice([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10]))
        base.dealer = [dealer_card, dealer_hides]
        if has_ace:
            base.player = [player_sum - 11, 1]        # ás vale 11 + outra carta
        elif player_sum == 21:
            base.player = [10, 9, 2]
        elif player_sum >= 12:
            base.player = [player_sum - 10, 10]
        else:
            base.player = [2, player_sum - 2]         # [ALTERADO] somas 4..11 (sem ás)
        soma = sum(base.player)
        tem_as = 1 in base.player and soma + 10 <= 21
        if tem_as:
            soma += 10
        return (soma, base.dealer[0], tem_as)

    def step(self, action):
        return self._env.step(action)


# Sanity check: o wrapper realmente reproduz todos os estados pedidos?
_chk = BlackjackES(seed=SEED)
_erros = [s for s in _chk.estados if _chk.reset_es(s) != s]
VERIF["estados_no_espaco_ES"] = len(_chk.estados)
VERIF["reset_es_divergencias"] = len(_erros)
print(f"Estados no espaço do ES: {len(_chk.estados)} | divergências em reset_es: {len(_erros)}")


def gerar_episodio_ES(env_es: BlackjackES, politica: Dict, S0, A0):
    """Episódio começando EXATAMENTE em (S0, A0); a partir do 2º passo, age pela política aprendida."""
    trajectory, done, At = [], True, None
    while True:
        if done:
            St = env_es.reset_es(S0)
            Rt, done, At = None, False, A0           # primeira ação forçada
        else:
            St, Rt, terminated, truncated, _ = env_es.step(At)
            done = bool(terminated or truncated)
            At = politica.get(St, STICK)             # política arbitrária inicial: STICK
        trajectory.append((St, Rt, done, At))
        if done:
            break
    return trajectory, len(trajectory) - 1

In [ ]:
def monte_carlo_ES_control(env_es: BlackjackES, episodes: int = 100_000, gamma: float = 1.0,
                           eval_env=None, checkpoints=(), n_eval: int = 10_000):
    """Monte Carlo ES (S&B 5.3). Retorna Q, pi e a curva de aprendizado
    [(episódios, retorno médio da política greedy)] medida em `checkpoints` com `eval_env`."""
    Q = defaultdict(float)
    Returns_sum = defaultdict(float)
    Counts = defaultdict(int)
    pi: Dict = {}
    curva = []
    cps = set(checkpoints)

    for ep in range(episodes):
        S0, A0 = env_es.amostrar_estado_acao()                       # 1) exploring start
        traj, T = gerar_episodio_ES(env_es, pi, S0, A0)              # 2) episódio a partir de (S0, A0)
        primeira = primeiras_visitas(traj, T, por_par=True)          # [ALTERADO]
        G = 0.0
        for t in range(T - 1, -1, -1):                               # 3) retornos, de trás para frente
            St, _, _, At = traj[t]
            G = gamma * G + traj[t + 1][1]
            sa = (St, At)
            if primeira[sa] == t:                                    # primeira visita ao par (s,a)
                Counts[sa] += 1
                Returns_sum[sa] += G
                Q[sa] = Returns_sum[sa] / Counts[sa]                 # média dos retornos
                q_vals = [Q.get((St, a), 0.0) for a in range(env_es.nb_actions)]
                pi[St] = argmax_aleatorio(q_vals, env_es.rng)        # 4) melhoria greedy
        if (ep + 1) in cps and eval_env is not None:
            curva.append((ep + 1, simular_politica(eval_env, politica_de_dict(pi), n_eval)["retorno_medio"]))
    return Q, pi, curva

## 5. On-Policy First-Visit MC Control com ε-greedy

**Teoria (S&B 5.4):** em vez de forçar o início, usa-se uma política **ε-soft**: com prob. $1-\varepsilon$ **aproveita** a melhor ação
conhecida e com prob. $\varepsilon$ **explora** uma ação aleatória, de modo que $\pi(a|s)\ge \varepsilon/|\mathcal{A}|$ para toda ação.
A melhoria de política é *implícita* (o próximo episódio já usa o $Q$ atualizado).
Como a política continua ε-soft, o método converge para a **melhor política ε-soft**, não para $\pi_*$ — por isso avaliamos a política **greedy** derivada de $Q$.

In [ ]:
def epsilon_greedy(Q: Dict, estado, n_acoes: int, eps: float, rng) -> int:
    """Com prob. 1-ε age de forma gulosa (aproveitamento); com prob. ε age ao acaso (exploração)."""
    q_vals = [Q.get((estado, a), 0.0) for a in range(n_acoes)]
    melhor = argmax_aleatorio(q_vals, rng)
    return melhor if rng.random() > eps else int(rng.integers(n_acoes))   # [ALTERADO] rótulos corrigidos


def gerar_episodio_on_policy(env, Q: Dict, eps: float, n_acoes: int, rng):
    """Episódio em que TODA ação é ε-greedy (o agente nunca manipula o ambiente)."""
    trajectory, done, At = [], True, None
    while True:
        if done:
            St, _ = env.reset()
            Rt, done = None, False
        else:
            St, Rt, terminated, truncated, _ = env.step(At)
            done = bool(terminated or truncated)
        At = epsilon_greedy(Q, St, n_acoes, eps, rng)
        trajectory.append((St, Rt, done, At))
        if done:
            break
    return trajectory, len(trajectory) - 1


def eps_linear(eps0: float, eps_min: float, episodios_ate_min: int):
    """[NOVO] Decaimento linear (Exercício 4): ε_t = max(ε_min, ε0 − t·Δε)."""
    delta = (eps0 - eps_min) / max(1, episodios_ate_min)
    return lambda t: max(eps_min, eps0 - t * delta)


def on_policy_first_visit_MC_control(env, episodes: int = 100_000, gamma: float = 1.0, eps=0.1, seed=None,
                                     eval_env=None, checkpoints=(), n_eval: int = 10_000):
    """On-Policy First-Visit MC Control. `eps` pode ser número (fixo) ou função t -> ε.
    Retorna Q e a curva de aprendizado medida em `checkpoints`."""
    rng = np.random.default_rng(seed)
    Q = defaultdict(float)
    Returns_sum = defaultdict(float)
    Counts = defaultdict(int)
    n_acoes = env.action_space.n
    curva, cps = [], set(checkpoints)

    for ep in range(episodes):
        eps_ep = eps(ep) if callable(eps) else eps
        traj, T = gerar_episodio_on_policy(env, Q, eps_ep, n_acoes, rng)
        primeira = primeiras_visitas(traj, T, por_par=True)          # [ALTERADO]
        G = 0.0
        for t in range(T - 1, -1, -1):
            St, _, _, At = traj[t]
            G = gamma * G + traj[t + 1][1]
            sa = (St, At)
            if primeira[sa] == t:
                Counts[sa] += 1
                Returns_sum[sa] += G
                Q[sa] = Returns_sum[sa] / Counts[sa]
        if (ep + 1) in cps and eval_env is not None:
            pol = politica_de_dict(derivar_politica_greedy(Q, n_acoes, rng))
            curva.append((ep + 1, simular_politica(eval_env, pol, n_eval)["retorno_medio"]))
    return Q, curva

## 6. Experimentos: ES × On-Policy ε-greedy com parâmetros diferentes

**Protocolo:** cada configuração roda com `N_SEEDS` sementes; em *checkpoints* de episódios medimos o retorno médio da política greedy
(`N_EVAL` jogos em ambiente separado) e, ao final, avaliamos com `N_FINAL` jogos. Resultados são médias ± erro padrão entre seeds.

| Experimento | Varia | Fixa |
|---|---|---|
| **B** | algoritmo (ES vs On-Policy) e **ε** ∈ {0.01, 0.05, 0.1, 0.2, 0.4}, nº de episódios (checkpoints) | γ=1 |
| **C** | **decaimento linear de ε** vs ε fixo | γ=1 |
| **D** | **γ** ∈ {1.0, 0.9} | ε=0.1 |

In [ ]:
if MODO_RAPIDO:
    EPISODIOS_MAX, N_SEEDS, N_EVAL, N_FINAL = 20_000, 2, 2_000, 5_000
    CHECKPOINTS = [1_000, 2_000, 5_000, 10_000, 20_000]
else:
    EPISODIOS_MAX, N_SEEDS, N_EVAL, N_FINAL = 200_000, 3, 10_000, 50_000
    CHECKPOINTS = [5_000, 10_000, 20_000, 50_000, 100_000, 200_000]

EPS_GRID = [0.01, 0.05, 0.1, 0.2, 0.4]
SEEDS = [SEED + 100 * i for i in range(N_SEEDS)]

rows_final, rows_curva, modelos = [], [], {}


import pickle, hashlib
CACHE_DIR = os.environ.get("MC_CACHE")   # opcional: guarda resultados parciais em disco (permite retomar execuções longas)


def _cache(chave, fn):
    if not CACHE_DIR:
        return fn()
    os.makedirs(CACHE_DIR, exist_ok=True)
    caminho = os.path.join(CACHE_DIR, hashlib.md5(f"{MODO_RAPIDO}|{chave}".encode()).hexdigest() + ".pkl")
    if os.path.exists(caminho):
        with open(caminho, "rb") as f:
            return pickle.load(f)
    r = fn()
    with open(caminho, "wb") as f:
        pickle.dump(r, f)
    return r


def executar(label, alg, seed, gamma=1.0, eps=None, episodes=EPISODIOS_MAX):
    """Treina uma configuração, mede a curva de aprendizado e o desempenho final."""
    t0 = time.time()

    def _treinar():
        eval_env = make_env(seed + 10_000)
        if alg == "ES":
            env_es = BlackjackES(seed=seed)
            Q, pi, curva = monte_carlo_ES_control(env_es, episodes, gamma, eval_env=eval_env,
                                                  checkpoints=CHECKPOINTS, n_eval=N_EVAL)
        else:
            env = make_env(seed)
            Q, curva = on_policy_first_visit_MC_control(env, episodes, gamma, eps, seed=seed,
                                                        eval_env=eval_env, checkpoints=CHECKPOINTS, n_eval=N_EVAL)
            pi = derivar_politica_greedy(Q, 2, np.random.default_rng(seed))
        res = simular_politica(make_env(seed + 20_000), politica_de_dict(pi), N_FINAL)
        return res, curva, dict(Q), dict(pi)

    res, curva, Q, pi = _cache(f"exec|{label}|{seed}|{gamma}|{episodes}", _treinar)
    rows_final.append({"label": label, "algoritmo": alg, "seed": seed, "gamma": gamma,
                       "eps": eps if not callable(eps) else "linear", **res})
    rows_curva.extend({"label": label, "seed": seed, "episodios": e, "retorno": r} for e, r in curva)
    modelos[(label, seed)] = (Q, pi)
    print(f"  {label:<26} seed={seed:<4} retorno_final={res['retorno_medio']:+.4f}  ({time.time() - t0:5.1f}s)")

### Experimento B — ES vs On-Policy para diferentes ε

In [ ]:
print("Experimento B")
for sd in SEEDS:
    executar("ES", "ES", sd)
    for e in EPS_GRID:
        executar(f"On-Policy ε={e}", "OP", sd, eps=e)

### Experimento C — decaimento linear de ε (Exercício 4 do professor)

In [ ]:
print("Experimento C")
for sd in SEEDS:
    executar("On-Policy ε linear 0.5→0.01", "OP", sd,
             eps=eps_linear(0.5, 0.01, int(0.8 * EPISODIOS_MAX)))

### Experimento D — efeito do fator de desconto γ

In [ ]:
print("Experimento D")
for sd in SEEDS:
    executar("ES γ=0.9", "ES", sd, gamma=0.9)
    executar("On-Policy ε=0.1 γ=0.9", "OP", sd, gamma=0.9, eps=0.1)

### Experimento E — efeito de ignorar os estados com soma < 12 (como no notebook original)

Compara o **ES original** (sorteia só somas 12–21 e usa STICK nos estados não aprendidos), o mesmo ES com `HIT` abaixo de 12
e o **ES corrigido** (sorteia os 280 estados) usado nos experimentos acima.

In [ ]:
print("Experimento E")
linhas_e = []
for sd in SEEDS:
    env_es12 = BlackjackES(seed=sd, soma_min=12)
    pi12 = _cache(f"expE|{sd}|{EPISODIOS_MAX}", lambda: dict(monte_carlo_ES_control(env_es12, EPISODIOS_MAX, 1.0)[1]))
    a = simular_politica(make_env(sd + 20_000), lambda s: pi12.get(s, STICK), N_FINAL)
    b = simular_politica(make_env(sd + 20_000), politica_de_dict(pi12), N_FINAL)
    c = [r for r in rows_final if r["label"] == "ES" and r["seed"] == sd][0]
    linhas_e += [("ES original (sorteia 12–21; STICK nos demais)", a["retorno_medio"]),
                 ("ES sorteia 12–21, mas HIT abaixo de 12", b["retorno_medio"]),
                 ("ES corrigido (sorteia 4–21)", c["retorno_medio"])]
tab_e = (pd.DataFrame(linhas_e, columns=["Variante", "retorno_medio"])
         .groupby("Variante", sort=False)["retorno_medio"].agg(["mean", "std"]).round(4))
print(tab_e.to_string())
salvar_csv(tab_e.reset_index(), "exp_E_vies_somas_baixas")

### Resultados consolidados

In [ ]:
df_final = pd.DataFrame(rows_final)
df_curva = pd.DataFrame(rows_curva)

resumo = (df_final.groupby("label", sort=False)
          .agg(retorno_medio=("retorno_medio", "mean"), desvio_entre_seeds=("retorno_medio", "std"),
               vitorias=("vitorias_%", "mean"), empates=("empates_%", "mean"), derrotas=("derrotas_%", "mean"))
          .round(4))
print(resumo.to_string())
salvar_csv(resumo.reset_index(), "resumo_experimentos")
salvar_csv(df_final, "experimentos_final_por_seed")
salvar_csv(df_curva, "experimentos_curvas")

In [ ]:
# Curvas de aprendizado: ES vs On-Policy (ε fixos e decaimento), com as políticas fixas como referência
grupo_B = ["ES"] + [f"On-Policy ε={e}" for e in EPS_GRID] + ["On-Policy ε linear 0.5→0.01"]
plot_curvas_aprendizado(
    df_curva[df_curva["label"].isin(grupo_B)],
    linhas_ref={"Depende da carta do dealer": REF_RETORNOS["Depende da carta do dealer"],
                "Simples ≥20": REF_RETORNOS["Simples ≥20 (professor)"]},
    titulo="Aprendizado: ES × On-Policy ε-greedy (média ± erro padrão entre seeds)",
    arquivo="exp_curvas_aprendizado")

In [ ]:
# Efeito de ε no desempenho final
op = df_final[df_final["label"].str.match(r"^On-Policy ε=[\d.]+$")].copy()
op["eps"] = op["eps"].astype(float)
agg = op.groupby("eps")["retorno_medio"].agg(["mean", "sem"])
es_medio = df_final.loc[df_final["label"] == "ES", "retorno_medio"].mean()

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.errorbar(agg.index, agg["mean"], yerr=agg["sem"].fillna(0), marker="o", capsize=3, label="On-Policy (greedy final)")
ax.axhline(es_medio, color="tab:red", ls="--", label="ES")
ax.axhline(REF_RETORNOS["Depende da carta do dealer"], color="gray", ls=":", label="Depende da carta do dealer (ref.)")
ax.set_xscale("log")
ax.set_xlabel("ε")
ax.set_ylabel("Retorno médio da política greedy final")
ax.set_title("Efeito de ε (On-Policy) × ES")
ax.legend()
fig.tight_layout()
salvar_fig(fig, "exp_efeito_epsilon")
plt.show()

In [ ]:
# Política e V(s) aprendidos (primeira seed) — ES e On-Policy ε=0.1
sd0 = SEEDS[0]
Q_es, pi_es = modelos[("ES", sd0)]
Q_op, pi_op = modelos[("On-Policy ε=0.1", sd0)]
plot_politica_heatmap(pi_es, "MC ES — política aprendida", arquivo="politica_es")
plot_politica_heatmap(pi_op, "On-Policy (ε=0.1) — política greedy aprendida", arquivo="politica_onpolicy_eps01")
plot_valor_heatmap(q_para_v(Q_es), "MC ES — V(s) ≈ max Q(s,a)", arquivo="v_es")
plot_valor_heatmap(q_para_v(Q_op), "On-Policy (ε=0.1) — V(s) ≈ max Q(s,a)", arquivo="v_onpolicy_eps01")

In [ ]:
# Quantas decisões diferem entre as políticas aprendidas? (somas 12–21)
f_es, f_op, f_ref = politica_de_dict(pi_es), politica_de_dict(pi_op), politica_dealer_fraco
estados = [(ps, dc, ace) for ps in range(12, 22) for dc in range(1, 11) for ace in (True, False)]
dif_es_op = np.mean([f_es(s) != f_op(s) for s in estados]) * 100
dif_es_ref = np.mean([f_es(s) != f_ref(s) for s in estados]) * 100
dif_op_ref = np.mean([f_op(s) != f_ref(s) for s in estados]) * 100
print(f"Divergência ES × On-Policy:                         {dif_es_op:5.1f}% dos estados")
print(f"Divergência ES × Depende da carta do dealer:        {dif_es_ref:5.1f}% dos estados")
print(f"Divergência On-Policy × Depende da carta do dealer: {dif_op_ref:5.1f}% dos estados")
salvar_csv(pd.DataFrame({"comparacao": ["ES x On-Policy", "ES x Depende da carta do dealer", "On-Policy x Depende da carta do dealer"],
                         "divergencia_%": [dif_es_op, dif_es_ref, dif_op_ref]}), "divergencia_politicas")

### Exportar V(s) e políticas aprendidas (tabelas usadas nos slides)

In [ ]:
def v_para_df(V, nome):
    return pd.DataFrame([{"modelo": nome, "soma": s[0], "dealer": s[1], "as_usavel": int(s[2]), "valor": v}
                         for s, v in V.items()])

df_v = pd.concat([v_para_df(V_fv[n], f"FV · {n}") for n in POLITICAS] +
                 [v_para_df(q_para_v(Q_es), "ES (seed 0)"), v_para_df(q_para_v(Q_op), "On-Policy ε=0.1 (seed 0)")])
salvar_csv(df_v, "valores_V_por_estado")

est_280 = [(ps, dc, ace) for ace in (False, True) for ps in range(12 if ace else 4, 22) for dc in range(1, 11)]
df_pol = pd.DataFrame([{"soma": s[0], "dealer": s[1], "as_usavel": int(s[2]),
                        "ES": f_es(s), "On-Policy ε=0.1": f_op(s), "Depende da carta do dealer": f_ref(s)} for s in est_280])
salvar_csv(df_pol, "politicas_aprendidas")
print(f"V(s): {len(df_v)} linhas · políticas: {len(df_pol)} estados")

## 7. Tabela-resumo para os slides
O arquivo `resultados/resumo_experimentos.csv` e as figuras `resultados/*.png` já estão prontos para a apresentação.

In [ ]:
salvar_csv(pd.DataFrame(list(VERIF.items()), columns=["verificacao", "valor"]), "verificacoes")
print(f"Arquivos gerados em {RES_DIR.resolve()}:")
for p in sorted(RES_DIR.glob("*")):
    print("  ", p.name)

## 8. Análise da implementação × teoria (rascunho — complete com as suas conclusões)

### 8.1 Mapa código ↔ teoria (S&B cap. 5)

| Trecho do código | Conceito teórico |
|---|---|
| `generate_episode` (trajetória `(S_t, R_t, done, A_t)`) | Episódio $S_0,A_0,R_1,\dots,S_T$ gerado pela política; MC só aprende de episódios **completos** |
| `G = gamma * G + traj[t+1][1]` em laço decrescente | Recorrência $G_t = R_{t+1}+\gamma G_{t+1}$ |
| `primeiras_visitas` / `if primeira[St] == t` | Critério **First-Visit** (estimador não viesado de $V_\pi(s)$, S&B 5.1) |
| `Returns_sum / Counts` | Média amostral dos retornos = $\mathbb{E}[G_t\mid s]$ (Lei dos Grandes Números); forma eficiente da lista `Returns(s)` |
| `amostrar_estado_acao` + `reset_es` | **Exploring Starts**: todo par $(s,a)$ com probabilidade > 0 de ser o início |
| `pi[St] = argmax_a Q(St,a)` dentro do laço (ES) | **Melhoria gulosa** — GPI: avaliação e melhoria intercaladas |
| `epsilon_greedy` | Política **ε-soft**: $\pi(a^*|s)=1-\varepsilon+\varepsilon/|\mathcal{A}|$ |
| ausência de atualização explícita de π (On-Policy) | Melhoria **implícita**: a ε-greedy sobre o $Q$ novo já é a política melhor |
| `Q.get(..., 0.0)` com retornos majoritariamente negativos | **Inicialização otimista**: ações não testadas parecem melhores → incentiva exploração |
| `derivar_politica_greedy` | A política ótima ε-soft ≠ $\pi_*$; extraímos a greedy para avaliar |

### 8.2 Perguntas para discutir (responda com os resultados da seção 6)

1. Por que ES é inviável fora de simuladores? (precisa controlar o estado inicial)
2. ES precisou de mais episódios que o On-Policy para estabilizar? A curva de aprendizado mostra isso?
3. Qual ε foi o melhor? Por que ε muito pequeno aprende devagar e ε muito grande piora o aprendizado?
4. O decaimento de ε superou o ε fixo? Em que ponto da curva?
5. Como γ=0.9 mudou a política (ações com episódios mais longos, como HIT, são penalizadas)? Por que γ=1 é o natural no Blackjack?
6. Políticas aprendidas × "Depende da carta do dealer": onde divergem e por quê? (ruído amostral em estados raros? ausência de *double/split*?)
7. O que muda com First-Visit vs Every-Visit aqui? (nada — veja seção 3.2)

### 8.3 Conclusões do grupo
_(escrever)_